# 03 — GraphConv (GCN) Classifier

학습 → 평가 → per-task ROC curve 12개. 결과는 `results/figures/` 와 Drive 에 저장.

**실행 순서**: 위에서부터, 또는 `런타임 > 모두 실행`. (STEP1 설치 → STEP2 Drive 순서 유지)

In [ ]:
# === STEP 1: 환경 설정 (가장 먼저 이 셀 실행) ===========================
import os, sys, subprocess

REPO = "Tox21-Toxicity-Classifier"
REPO_URL = "https://github.com/zqvo04/Tox21-Toxicity-Classifier.git"
ROOT = "/content/" + REPO

# (1) 레포 준비: 없으면 clone, 이미 있으면 최신으로 git pull(오래된 캐시 방지)
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", REPO_URL, ROOT], check=False)
else:
    subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
print("작업 디렉토리:", os.getcwd())

# (2) 의존성 설치 (idempotent). 'rdkit'(공식) 사용 — 'rdkit-pypi'는 최신
#     Python에서 설치 불가. 데이터/특징화는 RDKit으로 직접 처리(견고).
pkgs = ["rdkit", "torch-geometric",
        "pandas", "scikit-learn", "matplotlib", "seaborn", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# (3) 코드 버전 검증: 최신 RDKit 기반 dataset.py 인지 확인 (deepchem 의존 없음)
import importlib
import src.dataset as _ds
importlib.reload(_ds)
if not hasattr(_ds, "load_tox21_dataframe"):
    raise RuntimeError(
        "⚠️ 오래된 dataset.py 입니다 (deepchem 버전). 다음을 확인하세요:\n"
        "  1) 최신 코드를 GitHub 에 push 했는지\n"
        "  2) Colab 상단 [런타임 > 세션 다시 시작 및 모두 삭제] 후 이 셀을 다시 실행\n"
        "     (오래된 clone 폴더가 남아 있으면 갱신이 안 됩니다)")

# (4) import 점검. 실패 시 런타임 재시작 안내.
try:
    import rdkit, torch, torch_geometric, sklearn
    print("✅ rdkit", rdkit.__version__,
          "| torch", torch.__version__, "| pyg", torch_geometric.__version__,
          "| gpu", torch.cuda.is_available(), "| dataset.py: 최신(RDKit) ✔")
except Exception as e:
    print("⚠️ import 실패:", repr(e))
    print("➡️  상단 메뉴 [런타임 > 런타임 다시 시작] 후, 이 셀을 한 번 더 실행하세요.")

In [ ]:
# === STEP 2: Google Drive 마운트 (체크포인트/결과 저장) ==================
# 경로를 본인 Drive 폴더로 바꿔도 됩니다. 02·03·04 에서 동일하게 유지하세요.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/tox21/'
except Exception as e:
    # Colab이 아닌 환경이면 로컬 폴더로 대체
    print("Drive 마운트 생략 (로컬 폴더 사용):", repr(e))
    CKPT_DIR = './checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
print("CKPT_DIR =", CKPT_DIR)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from src import losses, models, train, evaluate
FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

## 1. 데이터 로딩

In [ ]:
from src.dataset import make_graph_dataloaders
train_loader, valid_loader, test_loader, tasks, node_dim = make_graph_dataloaders(batch_size=64)
print('tasks:', tasks, '| node_dim:', node_dim)

## 2. 손실함수 (pos_weight 자동 계산) & 모델

In [ ]:
from src.dataset import load_tox21_graph
_, (tr_ds, _, _), _ = load_tox21_graph()
pos_weight = losses.compute_pos_weight(tr_ds.y, tr_ds.w)
print('pos_weight:', np.round(pos_weight.numpy(), 2))
loss_fn = losses.build_loss('bce', pos_weight=pos_weight)

In [ ]:
model = models.build_model('gcn', node_dim=node_dim, hidden=128, n_tasks=len(tasks), dropout=0.3)
print(model)

## 3. 학습 (Early stopping, patience=10)

In [ ]:
cfg = train.TrainConfig(ckpt_dir=CKPT_DIR, ckpt_name='graphconv_best.pt',
                        max_epochs=100, patience=10, lr=1e-3)
trainer = train.Trainer(model, loss_fn, train.graph_forward, config=cfg, device=device)
history = trainer.fit(train_loader, valid_loader)
trainer.save_curves(f'{FIG}/graphconv_learning_curve.png', title='graphconv learning curve')

## 4. 평가 (test set): per-task ROC-AUC / PR-AUC / F1

In [ ]:
probs, y_true, mask = trainer.predict(test_loader)
df_metrics, summary = evaluate.evaluate(y_true, probs, mask, tasks, name='graphconv')
print('Summary:', summary)
df_metrics.to_csv(f'{FIG}/graphconv_metrics.csv', index=False)
display(df_metrics.round(4))

## 5. ROC curve 12개

In [ ]:
evaluate.plot_roc_curves(y_true, probs, mask, tasks,
                         save_path=f'{FIG}/graphconv_roc_curves.png',
                         title='graphconv: per-task ROC')
plt.show(); print('Mean ROC-AUC: %.4f' % summary['mean_roc_auc'])

## 6. 예측 저장 (노트북 04 비교용)

In [ ]:
np.savez(os.path.join(CKPT_DIR, 'graphconv_preds.npz'),
         probs=probs, y_true=y_true, mask=mask, tasks=np.array(tasks, dtype=object))
print('saved ->', os.path.join(CKPT_DIR, 'graphconv_preds.npz'))